[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rehanabbaxi/ocr-scaling-laws-label-noise/blob/main/notebook.ipynb)

# Scaling Laws for OCR under Label Noise — training driver

This notebook runs on **Google Colab with a GPU**. It holds no research logic: everything
substantive lives in `src/` in the repository, under test. The notebook only sets up the
machine, launches runs, and displays results.

**Before the first run**

The corpus archive `devanagari_lines.zip` (485 MB) must sit in your Drive at
`MyDrive/Projects/label_Noise_Project/`. That is the entire setup — the repository is
public, so no GitHub token is required.

**Runtime:** Runtime → Change runtime type → **T4 GPU**.


## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive

Drive holds two things: the corpus archive (input) and the run outputs
(checkpoints and results). Everything written to Drive survives a disconnect;
everything on the Colab disk does not.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Projects/label_Noise_Project'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive folder contents:', os.listdir(DRIVE_DIR))

## 3. Get the code

Clones the public repository. Re-running this cell pulls the latest commit rather than
cloning twice, so you can push a fix from your laptop and pick it up here without
restarting the runtime.


In [ ]:
import os, subprocess

REPO = 'https://github.com/Rehanabbaxi/ocr-scaling-laws-label-noise.git'
CODE_DIR = '/content/repo'

if os.path.isdir(CODE_DIR):
    print(subprocess.run(['git', '-C', CODE_DIR, 'pull', '--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(['git', 'clone', REPO, CODE_DIR],
                   check=True, capture_output=True, text=True)
    print('cloned')

print(subprocess.run(['git', '-C', CODE_DIR, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)


## 4. Install the one missing dependency

Colab already ships torch, pandas, Pillow and lxml. Only the edit-distance
library used by the CER metric is absent.

In [ ]:
!pip install -q editdistance

## 5. Stage the corpus on local disk

Reading 5,053 small files directly from Drive is extremely slow — Drive is a network
filesystem and each file costs a round trip. The archive is copied once as a single
508 MB file, then unzipped onto Colab's local disk, and training reads from there.

This takes roughly 2–4 minutes and must be repeated after every runtime restart.

In [ ]:
import os, shutil, time, zipfile

WORK_ROOT = '/content/work'
LINES_DIR = f'{WORK_ROOT}/data/lines'
ZIP_SRC   = f'{DRIVE_DIR}/devanagari_lines.zip'

os.makedirs(LINES_DIR, exist_ok=True)
expected = 5053
have = len(os.listdir(f'{LINES_DIR}/devanagari')) if os.path.isdir(f'{LINES_DIR}/devanagari') else 0

if have >= expected:
    print(f'already staged: {have} crops')
else:
    t0 = time.time()
    local_zip = '/content/devanagari_lines.zip'
    if not os.path.exists(local_zip):
        shutil.copy(ZIP_SRC, local_zip)
        print(f'copied from Drive in {time.time()-t0:.0f}s')

    with zipfile.ZipFile(local_zip) as z:
        z.extractall(LINES_DIR)

    n = len(os.listdir(f'{LINES_DIR}/devanagari'))
    print(f'staged {n} crops in {time.time()-t0:.0f}s total')
    assert n >= expected, f'expected at least {expected} crops, found {n}'

## 6. Point the pipeline at these directories

`src/config.py` reads `OCR_WORK_ROOT` and `OCR_PERSISTENT_ROOT` **when it is first
imported**, so these must be set before any import from `src`. If you change them
later, restart the runtime.

- **work root** → Colab local disk: the line crops (fast, disposable)
- **persistent root** → Drive: manifests, checkpoints, results (durable)

In [ ]:
import os, sys, shutil

os.environ['OCR_WORK_ROOT'] = '/content/work'
os.environ['OCR_PERSISTENT_ROOT'] = f'{DRIVE_DIR}/output'

sys.path.insert(0, '/content/repo/src')

# The manifests are version-controlled in the repo; copy them to the persistent
# root, which is where the pipeline expects to read them from.
os.makedirs(f"{os.environ['OCR_PERSISTENT_ROOT']}/manifests", exist_ok=True)
for name in ('devanagari_lines.csv', 'devanagari_drops.csv', 'devanagari_vocab.json'):
    shutil.copy(f'/content/repo/manifests/{name}',
                f"{os.environ['OCR_PERSISTENT_ROOT']}/manifests/{name}")

from config import CONFIG, PATHS, describe_device, make_dirs
make_dirs()

print(describe_device())
print()
for k in ('work_root', 'persistent_root', 'lines_dir', 'manifests_dir', 'results_dir'):
    print(f'{k:16s} {PATHS[k]}')

## 7. Sanity checks before spending GPU time

Each of these has caught a real bug at some point. Run them after any change to the
pipeline, not just the first time.

In [ ]:
import torch
from data import load_manifest, split_by_page, split_summary
from dataset import make_loader
from model import build_model, count_parameters
from vocab import build_vocab

df = load_manifest('devanagari')
splits = split_by_page(df, CONFIG['val_split'], CONFIG['test_split'], CONFIG['seed'])
vocab = build_vocab(splits['train']['text'])

print(split_summary(splits).to_string(index=False))
print(f'\nclasses: {vocab.num_classes}  (C={vocab.n_chars}, V={vocab.size})')

# Pages must never appear in two splits: that would leak test data into training.
for a in ('train', 'val', 'test'):
    for b in ('train', 'val', 'test'):
        if a < b:
            assert not set(splits[a].page_id) & set(splits[b].page_id), f'{a}/{b} overlap'
print('page disjointness: OK')

# One batch, end to end, with the real model.
loader = make_loader(splits['val'], PATHS['lines_dir'] / 'devanagari', vocab,
                     batch_size=8, shuffle=False, num_workers=2)
batch = next(iter(loader))
model = build_model(vocab.num_classes).cuda()
out = model(batch['images'].cuda())

print(f'\nbatch images    : {tuple(batch["images"].shape)}')
print(f'model output    : {tuple(out.shape)}  (T, B, classes)')
print(f'input_lengths   : {batch["input_lengths"].tolist()}')
print(f'parameters      : {count_parameters(model):,}')

# Every line must get more timesteps than it has characters, or CTC cannot align it.
assert (batch['input_lengths'] >= batch['target_lengths']).all(), 'CTC infeasible batch'
print('CTC feasibility : OK')

## 8. The baseline run

All the data, no injected noise — the easiest cell in the grid. Everything else is
this same call with two values changed.

**Success criterion: validation CER below roughly 30 %.** Below 10 % is good. The
~2.29 % that Heidelberg reported with Transkribus is not the target; that is a mature
production system.

On a T4 this takes roughly 20–35 minutes. Progress is written to Drive every epoch, so
a disconnect costs you the run but not the curve.

In [ ]:
from train import train

cfg = dict(CONFIG)
cfg.update({
    'data_fraction': 1.0,     # the scaling axis
    'noise_type':    'none',  # the noise axis
    'noise_rate':    0.0,
    'seed':          42,
    'num_workers':   2,       # Colab gives 2 CPU cores
})

result = train(cfg)

## 9. Read the predictions

A CER number alone will not tell you whether the model has learned the script or
merely learned to emit common characters. Read some output.

In [ ]:
import json

run_name = result['run_name']
examples = json.loads(
    (PATHS['results_dir'] / f'{run_name}_examples.json').read_text(encoding='utf-8')
)

for ex in examples['val'][:8]:
    print('ref :', ex['reference'])
    print('pred:', ex['prediction'])
    print('    ', 'exact match' if ex['reference'] == ex['prediction'] else '')
    print()

### Training curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.read_csv(PATHS['results_dir'] / f'{run_name}_history.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(hist['epoch'], hist['train_loss'])
ax1.set_xlabel('epoch'); ax1.set_ylabel('train CTC loss'); ax1.set_title('Training loss')

ax2.plot(hist['epoch'], hist['val_cer'], label='CER')
ax2.plot(hist['epoch'], hist['val_wer'], label='WER')
ax2.axhline(0.30, ls='--', c='grey', lw=1, label='success bar')
ax2.set_xlabel('epoch'); ax2.set_ylabel('rate'); ax2.set_title('Validation error'); ax2.legend()

for ax in (ax1, ax2):
    ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. The results table

Every run writes its own `results/<run_name>.json`. Nothing appends to a shared CSV,
so two people running experiments on separate branches never collide. This cell
collects them into one table.

In [ ]:
from train import aggregate_results

table = aggregate_results()
cols = ['run_name', 'data_fraction', 'noise_rate', 'n_train_lines',
        'val_cer', 'test_cer', 'best_epoch', 'epochs_run', 'minutes']
table[[c for c in cols if c in table]]

## 11. The sweep — only after the baseline passes

Sixteen cells: four data fractions by four noise rates. Do **not** run this until the
baseline has cleared the 30 % bar; otherwise you are fitting a curve through sixteen
broken runs.

Each cell writes its own result file, so the loop is safe to interrupt and resume —
already-completed runs are skipped.

In [ ]:
# from train import train, aggregate_results
# from config import build_run_name
#
# for fraction in (0.125, 0.25, 0.5, 1.0):
#     for rate in (0.0, 0.1, 0.2, 0.4):
#         cfg = dict(CONFIG)
#         cfg.update({
#             'data_fraction': fraction,
#             'noise_type':    'none' if rate == 0 else 'char_flip',
#             'noise_rate':    rate,
#             'num_workers':   2,
#         })
#         name = build_run_name(cfg)
#         if (PATHS['results_dir'] / f'{name}.json').exists():
#             print(f'skip {name} (already done)')
#             continue
#         train(cfg)
#
# aggregate_results()